# The stochastic O&M tool for down side risk in TIMES

## How to use
In your environment run 'pip install -e' to install the packages and dependensies

Then having the 'offshore_om' package you can import the nesessary functions and start.

In [1]:
from pathlib import Path

from offshore_om import run_master
from offshore_om import run_subsidy_capex
from offshore_om import load_scenario_configs

With the right depensesie intialise and cofigure your senario as you want with the right sensetivities and data
## Path configurations

In [2]:
# Root folder for the project. Update this if your workspace moves.
ROOT_DIR = Path(
    "C:/Users/IFE13253/OneDrive - Institutt for Energiteknikk/Documents/offshore-wind-om"
)

#Input and output folders
TIMES_INPUT_DIR = ROOT_DIR / "times-input"
OUTPUT_DIR = ROOT_DIR / "model-results"

Having cofigured the main paths you can configure the scenarios and the corresponding files from the input
## Scenario configuration
Her is an exemple with the inc and tech scenarios

In [3]:
# Scenario definitions for the master simulation.
# The Master script will read these files and optionally use market prices.
SCENARIO_CONFIGS = [
    {
        "Scenario": "TECH market",
        "MarketFile": TIMES_INPUT_DIR / "tech" / "tech onshore shadow price.csv",
        "production_folder": TIMES_INPUT_DIR / "tech",
        "market_price_file": TIMES_INPUT_DIR / "tech" / "tech_price.csv",
        "capacity_file": TIMES_INPUT_DIR / "tech" / "tech_capacity.csv",
    },
    {
        "Scenario": "INC market",
        "MarketFile": TIMES_INPUT_DIR / "inc" / "inc onshore shadow price.csv",
        "production_folder": TIMES_INPUT_DIR / "inc",
        "market_price_file": TIMES_INPUT_DIR / "inc" / "inc_price.csv",
        "capacity_file": TIMES_INPUT_DIR / "inc" / "inc_capacity.csv",
    },
]

In [4]:
# TIMES subsidy input and output configuration.
# This config only contains folder/file locations and mappings.
# Discount rates and base year are global and are not set here.
TIMES_SUBSIDY_CONFIG = {
    "region_price_map": {
        "O_Sorvest": "NO2",
        "O_Vestavind2": "NO2",
        "O_Nordvest": "NO3",
        "O_Nordavind": "NO4",
        "O_Vestavind1": "NO5",
    },
    "file_pattern": "*.csv",
    "process_filter": None,
}

Having the different scenarios you want to model import the sensitivities
## Sensetivity lists

In [5]:
# Sensitivity lists to sweep in the master simulation.
DISCOUNT_RATE_LIST = [0.06, 0.07]  # e.g. [0.06, 0.07, 0.08]
TURBINE_CAPACITY_LIST = [15, 20]  # e.g. [12, 15, 17, 20]
N_TURBINES_LIST = [75 ,100]  # e.g. [1, 5, 10, 20, 35, 50, 75, 100]
HORIZON_YEARS = [25.0]  # e.g. [20.0, 25.0, 30.0]
FAILURE_RATE_TYPES = ["carrol"]  # e.g. ["carrol", "hendriks"]
SUBSIDY_PRICE_LIST = [100, 115]

base_year = 2030

total_runs = (len(DISCOUNT_RATE_LIST) * len(TURBINE_CAPACITY_LIST) * len(N_TURBINES_LIST) * len(HORIZON_YEARS) * len(FAILURE_RATE_TYPES) * len(SUBSIDY_PRICE_LIST) * len(SCENARIO_CONFIGS))

print(total_runs)

32


## Master table execution
Excecution of the montecarlo simulation and financial logic excecuted for all the sensitivites etc

In [8]:
master_df = run_master(
    scenario_configs=load_scenario_configs(SCENARIO_CONFIGS),
    turbine_capacity_list=TURBINE_CAPACITY_LIST,
    n_turbines_list=N_TURBINES_LIST,
    discount_rate_list=DISCOUNT_RATE_LIST,
    horizon_years=HORIZON_YEARS,
    failure_rate_types=FAILURE_RATE_TYPES,
    subsidy_price_list=SUBSIDY_PRICE_LIST,
    output_dir=OUTPUT_DIR,
)



Simulating for region: Nordavind

Completed simulation 1/1000
Campaigns: 49
Repairs: 72

Completed simulation 1000/1000
Campaigns: 54
Repairs: 80

SIMULATION SUMMARY
Mean Campaigns: 48.913
Mean direct cost: 5689717.711876281
Mean lost MWh: 873954.0757208487
Installed capacity: 1125
Mean downtime per turbine: 1572.5669378692726

AVERAGE REPAIRS PER TURBINE OVER HORIZON
Gearbox   0.821 repairs/lifetime
Blades    0.076 repairs/lifetime
Generator 0.072 repairs/lifetime

Completed simulation 1/1000
Campaigns: 61
Repairs: 99

Completed simulation 1000/1000
Campaigns: 60
Repairs: 96

SIMULATION SUMMARY
Mean Campaigns: 58.755
Mean direct cost: 7355072.8183055995
Mean lost MWh: 1169747.6394673332
Installed capacity: 1500
Mean downtime per turbine: 1578.6068009005842

AVERAGE REPAIRS PER TURBINE OVER HORIZON
Gearbox   0.820 repairs/lifetime
Blades    0.077 repairs/lifetime
Generator 0.073 repairs/lifetime
Simulating for region: Nordvest

Completed simulation 1/1000
Campaigns: 49
Repairs: 72

Com

## TIMES subsidy representation
This is the output to represent subsidy gains in TIMES

In [6]:
times_capex_df = run_subsidy_capex(
    times_subsidy_config=TIMES_SUBSIDY_CONFIG,
    discount_rate_list=DISCOUNT_RATE_LIST,
    subsidy_price_list=SUBSIDY_PRICE_LIST,
    times_scenario=SCENARIO_CONFIGS,
    base_year = base_year,
    output_dir=OUTPUT_DIR,
)